# spVIPESmulti gallery — Malaria B-cells

This notebook is a derivative of `malaria_bcells_recommended.ipynb`. It uses the **same dataset, preprocessing, and recommended training configuration** but exercises as much of the public `spVIPESmulti` API as possible to serve as a *gallery* and a *runtime smoke-test*.

Sections that differ from the recommended notebook are tagged with **✨ gallery** in the markdown headers and use package functions (`spVIPESmulti.utils.*`, `model.embed`, `model.evaluate`, `model.get_shared_posterior`, `model.differential_abundance`, `model.get_loadings`, `model.traverse_latent`, `model.get_enrichment_scores`, `spVIPESmulti.pl.*`, etc.) in place of inlined ad-hoc code.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

import scvi
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import spVIPESmulti

np.random.seed(0)
torch.manual_seed(0)
sc.settings.set_figure_params(dpi=80, frameon=False)

print(f"spVIPESmulti  : {spVIPESmulti.__version__}")
print(f"scvi-tools    : {scvi.__version__}")
print(f"scanpy        : {sc.__version__}")
print(f"torch         : {torch.__version__} (CUDA available: {torch.cuda.is_available()})")
print(f"anndata       : {ad.__version__}")

## 1. Load and tidy

In [ ]:
root = Path("/exports/para-lipg-hpc/mdmanurung/spVIPESmulti/docs/notebooks/")
path_obs = root / "data/bcells_obs.csv"
path_rna = root / "data/bcells_rna.csv"

adata = ad.read_csv(path_rna, first_column_names=True)
obs = pd.read_csv(path_obs, index_col=0)
adata.obs = obs
adata.layers["counts"] = adata.X.copy()

# Drop antigen-negative cells and clean up the antigen label.
adata = adata[adata.obs["antigen_specific"] != "Negative"].copy()
adata.obs["antigen_class"] = (
    adata.obs["antigen_specific"].astype(str).str.strip().str.capitalize()
)
adata

In [ ]:
for col in ["volunteer", "batch", "time", "cluster_label", "antigen_specific"]:
    print(f"── {col} ──")
    print(adata.obs[col].value_counts().to_string())
    print()

## 2. Highly variable genes — ✨ gallery (`utils.highly_variable_genes_union`)

Replaces the manual `sc.pp.highly_variable_genes(..., batch_key="batch")` call. The helper computes HVGs **per (group × batch)** and returns the union, which avoids dropping genes that vary in only one antigen group.

In [ ]:
adata = spVIPESmulti.utils.highly_variable_genes_union(
    adata,
    group_key="antigen_specific",
    n_top_genes=3000,
    flavor="seurat_v3",
    batch_key="batch",
    subset=True,
)
print(f"After HVG union: {adata.shape}")

## 3. Prepare multi-group AnnData (`data.prepare_adatas`)

In [ ]:
from spVIPESmulti.utils import resolve_group_indices_list

antigen_specific = sorted(adata.obs["antigen_specific"].unique())
adatas_dict = {}
for ac in antigen_specific:
    sub = adata[adata.obs["antigen_specific"] == ac].copy()
    sub.uns = {}
    sub.obsm = {}
    sub.layers = {}
    adatas_dict[ac] = sub
    print(f"  {ac}: {sub.shape}")

adata_spv = spVIPESmulti.data.prepare_adatas(adatas_dict)
group_indices_list, _ = resolve_group_indices_list(adata_spv)

print(f"\nConcatenated AnnData: {adata_spv.shape}")
print(f"Groups              : {list(adata_spv.uns['groups_mapping'].values())}")
print(f"Group sizes         : {[len(g) for g in group_indices_list]}")

In [ ]:
# Register groups, label, batch, AND sample (for posterior aggregation later).
spVIPESmulti.model.spVIPESmulti.setup_anndata(
    adata_spv,
    groups_key="antigen_specific",
    label_key="cluster_label",
    batch_key="batch",
    sample_key="volunteer",
)

## 4. Recommended training configuration

Identical to `malaria_bcells_recommended.ipynb` v4:

* `n_hidden=256`, `n_dimensions_shared=20`, `n_dimensions_private=20`
* `disentangle_preset="full"` with `label_shared_w=4`, `label_private_w=0.5`
* `use_jeffreys_integ=True`, `jeffreys_integ_weight=0.5`
* Inverse-sqrt group-loss weighting (corrects for `ConcatDataLoader` cycling)
* Cosine LR schedule, `lr=5e-4`, `batch_size=1024`, `max_epochs=400`
* Early stopping on `reconstruction_loss_validation` with patience 20

In [ ]:
N_SHARED   = 20
N_PRIVATE  = 20
N_HIDDEN   = 256
DROPOUT    = 0.1
MAX_EPOCHS = 400
BATCH_SIZE = 1024
KL_WARMUP  = 75
LABEL_SHARED_W  = 4
LABEL_PRIVATE_W = 0.5

GROUP_SIZES = [len(g) for g in group_indices_list]
GROUP_LOSS_WEIGHTS = [1 / n**0.5 for n in GROUP_SIZES]
print("Group sizes        :", dict(zip(adata_spv.uns["groups_mapping"].values(), GROUP_SIZES)))
print("Group loss weights :", [round(w, 4) for w in GROUP_LOSS_WEIGHTS])

model_spv = spVIPESmulti.model.spVIPESmulti(
    adata_spv,
    n_hidden=N_HIDDEN,
    n_dimensions_shared=N_SHARED,
    n_dimensions_private=N_PRIVATE,
    dropout_rate=DROPOUT,
    disentangle_preset="full",
    disentangle_label_shared_weight=LABEL_SHARED_W,
    disentangle_label_private_weight=LABEL_PRIVATE_W,
    use_jeffreys_integ=True,
    jeffreys_integ_weight=0.5,
    group_loss_weights=GROUP_LOSS_WEIGHTS,
)
model_spv

In [ ]:
import os
_save_dir = 'results/spvipes_bcells_gallery'
if os.path.exists(os.path.join(_save_dir, 'model.pt')):
    print('Loading saved model from', _save_dir)
    model_spv = spVIPESmulti.model.spVIPESmulti.load(_save_dir, adata=adata_spv)
else:
    model_spv.train(
        group_indices_list,
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        train_size=0.9,
        early_stopping=True,
        early_stopping_patience=20,
        early_stopping_monitor="reconstruction_loss_validation",
        n_epochs_kl_warmup=KL_WARMUP,
        check_val_every_n_epoch=5,
        plan_kwargs={
            "lr": 5e-4,
            "lr_scheduler_type": "cosine",
            "lr_min": 1e-5,
        },
    )
    model_spv.save(_save_dir, overwrite=True)


In [ ]:
# Training history table
_h = model_spv.history
print(f"{'metric':45s}  {'first':>10}  {'final':>10}  {'drop':>10}  {'epochs':>7}")
print("-" * 90)
for _key in [
    "reconstruction_loss_train",
    "reconstruction_loss_validation",
    "elbo_train",
    "kl_local_train",
]:
    if _key in _h and len(_h[_key]) > 0:
        _arr = np.asarray(
            [float(np.asarray(v).ravel()[0]) for v in _h[_key].to_numpy().ravel()],
            dtype=float,
        )
        _ok = np.isfinite(_arr).all()
        print(
            f"{_key:45s}  {_arr[0]:10.1f}  {_arr[-1]:10.1f}"
            f"  {_arr[0] - _arr[-1]:10.1f}  {len(_arr):7d}  {'OK' if _ok else 'NaN!'}"
        )
    else:
        print(f"{_key:45s}  (not recorded)")

In [ ]:
model_spv.save("results/spvipes_bcells_gallery", overwrite=True)

In [ ]:
# ✨ gallery — spVIPESmulti.pl.training_curves
fig = spVIPESmulti.pl.training_curves(model_spv)

## 5. Embed cells — ✨ gallery (`model.embed`)

`embed()` is a one-call replacement for `get_latent_representation` + `store_latents`. It writes:

* `adata.obsm["X_spvm_shared"]` (all cells, original order)
* `adata.obsm["X_spvm_private_<group_token>"]` (one per group; rows for cells in other groups are zero)

In [ ]:
embed_out = model_spv.embed(prefix="spvm", batch_size=1024, overwrite=True)
print("Written keys:", embed_out["keys"])
print("Shared shape:", embed_out["shared"].shape)
for tok, arr in embed_out["private"].items():
    print(f"Private[{tok}] shape: {arr.shape}")

## 6. Quantitative evaluation — ✨ gallery (`model.evaluate`)

`evaluate()` wraps `metrics.integration_report` and additionally pulls the latest validation losses from `model.history`. Asking for `include_private=True` runs per-group silhouette on each private latent.

In [ ]:
eval_out = model_spv.evaluate(
    label_key="cluster_label",
    z_shared_key="X_spvm_shared",
    k=20,
    leiden_resolution=0.8,
    include_private=True,
)
print("Metadata:", eval_out["metadata"])
print("\nHeld-out validation metrics:", eval_out["held_out_metrics"])
if eval_out["warnings"]:
    print("\nWarnings:")
    for w in eval_out["warnings"]:
        print(" •", w)
eval_out["metrics"]

## 7. Failure-mode audit (per-cell-type k-NN purity, AUROC)

Direct use of `sklearn` against `adata.obsm["X_spvm_shared"]` to flag cell types that are squashed in shared space.

In [ ]:
from sklearn.neighbors import NearestNeighbors as _NNS
from sklearn.metrics import roc_auc_score as _auc

z_shared = adata_spv.obsm["X_spvm_shared"]
_K = 20
_labels = adata_spv.obs["cluster_label"].values
_nn = _NNS(n_neighbors=_K + 1).fit(z_shared)
_, _idx = _nn.kneighbors(z_shared)
_idx = _idx[:, 1:]

_purity = np.array([(_labels[_idx[i]] == _labels[i]).mean() for i in range(len(_labels))])
_purity_df = (
    pd.DataFrame({"cluster_label": _labels, "knn_purity": _purity})
    .groupby("cluster_label")["knn_purity"]
    .agg(["mean", "std", "count"])
    .sort_values("mean")
)
print("Per-cell-type k-NN purity (k=20) in z_shared — lower = harder to separate:")
print(_purity_df.to_string(float_format=lambda x: f"{x:.3f}"))

_cell_types = sorted(np.unique(_labels))
_auroc = np.full((len(_cell_types), z_shared.shape[1]), np.nan)
for _ci, _ct in enumerate(_cell_types):
    _y = (_labels == _ct).astype(int)
    if _y.sum() < 5 or (1 - _y).sum() < 5:
        continue
    for _di in range(z_shared.shape[1]):
        try:
            _auroc[_ci, _di] = _auc(_y, z_shared[:, _di])
        except Exception:
            pass
_auroc_df = pd.DataFrame(
    _auroc, index=_cell_types,
    columns=[f"shared_{i}" for i in range(z_shared.shape[1])],
)
print("\nBest shared-dim one-vs-rest AUROC per cell type:")
print(_auroc_df.max(axis=1).sort_values().to_string(float_format=lambda x: f"{x:.3f}"))

## 8. Shared latent UMAP — ✨ gallery (`utils.compute_shared_umap` + `pl.umap_shared`)

In [ ]:
spVIPESmulti.utils.compute_shared_umap(
    adata_spv,
    obsm_key="X_spvm_shared",
    n_neighbors=20,
    min_dist=1.0,
    umap_key="X_umap_spvm_shared",
)
spVIPESmulti.pl.umap_shared(
    adata_spv,
    color=["cluster_label", "antigen_specific", "time", "volunteer"],
    basis="X_umap_spvm_shared",
)

## 9. Per-group private UMAPs — ✨ gallery (`utils.compute_private_umaps` + `pl.umap_private`)

In [ ]:
# Build per-group AnnData copies that carry the private latent.
groups_map = adata_spv.uns["groups_mapping"]
private_adatas = {}
for gi, idxs in enumerate(group_indices_list):
    name = str(groups_map.get(gi, gi))
    sub = adata_spv[np.asarray(idxs)].copy()
    private_key = embed_out["keys"]["private"][
        list(embed_out["keys"]["private"].keys())[gi]
    ]
    sub.obsm["X_spvm_private"] = adata_spv.obsm[private_key][np.asarray(idxs)]
    private_adatas[name] = sub

spVIPESmulti.utils.compute_private_umaps(
    private_adatas,
    obsm_key="X_spvm_private",
    n_neighbors=20,
    min_dist=0.5,
    umap_key="X_umap_spvm_private",
)
fig = spVIPESmulti.pl.umap_private(
    private_adatas,
    color="cluster_label",
    basis="X_umap_spvm_private",
    ncols=3,
)

## 10. Latent-dimension diagnostics — ✨ gallery

* `pl.plot_latent_dims_in_umap` — each shared dim painted on the UMAP.
* `pl.plot_latent_dims_in_heatmap` — mean activity per cell type × dimension.
* `pl.plot_latent_dimension_stats` — per-dimension std (flags inactive dims).
* `pl.factor_violin` — distribution of one factor across cell types.

In [ ]:
fig = spVIPESmulti.pl.plot_latent_dims_in_umap(
    adata_spv,
    obsm_key="X_spvm_shared",
    basis="X_umap_spvm_shared",
)

In [ ]:
fig = spVIPESmulti.pl.plot_latent_dims_in_heatmap(
    adata_spv,
    obsm_key="X_spvm_shared",
    groupby="cluster_label",
)

In [ ]:
_dim_stats = spVIPESmulti.metrics.latent_dimension_stats(adata_spv.obsm["X_spvm_shared"])
print(_dim_stats.head())
fig = spVIPESmulti.pl.plot_latent_dimension_stats(_dim_stats)

In [ ]:
# Pick the most-active shared dimension and visualize it across cell types.
_stds = adata_spv.obsm["X_spvm_shared"].std(axis=0)
_top_dim = int(np.argmax(_stds))
print(f"Most-active shared dim: {_top_dim} (std={_stds[_top_dim]:.3f})")
spVIPESmulti.pl.factor_violin(
    adata_spv,
    dim_idx=_top_dim,
    groupby="cluster_label",
    obsm_key="X_spvm_shared",
    latent_type="shared",
    rotation=90,
)

## 11. Score cells on a single factor — ✨ gallery (`utils.score_cells_on_factor`)

In [ ]:
spVIPESmulti.utils.score_cells_on_factor(
    adata_spv,
    dim_idx=_top_dim,
    obsm_key="X_spvm_shared",
    col_name=f"shared_factor_{_top_dim}",
)
sc.pl.embedding(
    adata_spv,
    basis="X_umap_spvm_shared",
    color=f"shared_factor_{_top_dim}",
    cmap="viridis",
)

## 12. Decoder loadings — ✨ gallery (`model.get_loadings`, `pl.heatmap_loadings`, `pl.loadings_dotplot`, `utils.get_top_genes`)

In [ ]:
loadings = model_spv.get_loadings()
print("Loadings keys:", list(loadings.keys()))
shared_loadings_g0 = loadings[(0, "shared")]
private_loadings_g0 = loadings[(0, "private")]
print("Shared loadings (group 0):", shared_loadings_g0.shape)
print("Private loadings (group 0):", private_loadings_g0.shape)

In [ ]:
top_shared = spVIPESmulti.utils.get_top_genes(
    shared_loadings_g0, n_top=8, signed=True,
)
top_shared.head(5)

In [ ]:
ax = spVIPESmulti.pl.heatmap_loadings(
    model=model_spv, group_idx=0, latent_type="shared", n_top=5,
)

In [ ]:
spVIPESmulti.pl.loadings_dotplot(
    adata_spv,
    model=model_spv,
    groupby="cluster_label",
    group_idx=0,
    latent_type="shared",
    n_top=4,
    dims=list(range(min(8, N_SHARED))),
)

## 13. Latent traversal — ✨ gallery (`model.traverse_latent`, `traversal.calculate_differential_vars`, `pl.show_top_differential_vars`, `pl.differential_vars_heatmap`)

Sweeps each shared dimension across ±3 SD and records how strongly each gene's expected expression changes.

In [ ]:
trav = model_spv.traverse_latent(group_idx=0, n_steps=11, n_samples=50, n_stds=3.0, seed=0)
print("Traversal effect matrix:", trav.shape)
trav.iloc[:5, :5]

In [ ]:
diff_vars = spVIPESmulti.traversal.calculate_differential_vars(trav)
diff_vars.head()

In [ ]:
spVIPESmulti.pl.show_top_differential_vars(
    diff_vars, dim_idx=_top_dim, top_n=15,
)

In [ ]:
spVIPESmulti.pl.differential_vars_heatmap(trav, top_n_genes=30)

## 14. Posterior aggregation — ✨ gallery (`model.get_shared_posterior` + `model.get_aggregated_posterior`)

Aggregates the shared posterior per (group, sample). Requires the `sample_key` registered above (`volunteer`).

In [ ]:
shared_post = model_spv.get_shared_posterior(batch_size=1024)
print("Per-group loc shapes:", {g: arr.shape for g, arr in shared_post["loc_reordered"].items()})

agg = model_spv.get_aggregated_posterior(batch_size=1024)
agg_df = agg["posterior"]
print(f"\nAggregated posterior: {agg_df.shape[0]} (group, sample) pairs")
agg_df.drop(columns=["loc", "scale"]).head(10)

## 15. Differential abundance — ✨ gallery (`model.differential_abundance`)

Per-cell signed score in the shared latent comparing two antigen groups (we use groups 0 vs 1).

In [ ]:
da = model_spv.differential_abundance(group_a=0, group_b=1, batch_size=1024)
print("Comparison:", da["metadata"]["group_a_name"], "vs", da["metadata"]["group_b_name"])
da_scores = da["scores"]
adata_spv.obs["da_score_g0_vs_g1"] = da_scores["da_score"].values
print("\nScore distribution by antigen group:")
print(adata_spv.obs.groupby("antigen_specific")["da_score_g0_vs_g1"].describe()[["mean", "std", "count"]])

sc.pl.embedding(
    adata_spv, basis="X_umap_spvm_shared",
    color="da_score_g0_vs_g1", cmap="RdBu_r", vcenter=0,
)

## 16. Pathway enrichment with PROGENy — ✨ gallery

Runs `decoupler` enrichment methods through `model.get_enrichment_scores`, summarises per cluster, and renders the dashboard. We use a small PROGENy network (top-200 targets per pathway).

In [ ]:
import decoupler as dc

progeny_net = dc.op.progeny(organism="human", top=200)
print("PROGENy network:", progeny_net.shape)
progeny_net.head()

In [ ]:
# Notes:
# • `adata_spv.var_names` are group-prefixed (e.g. 'Pf_TNF'), so PROGENy gene symbols
#   don't match → call decoupler directly on a bare-HGNC AnnData.
# • Use the HVG-filtered `adata` (cell 6) with the canonical gene names.
# • obs_names are non-unique across volunteers → align by positional permutation.
import numpy as _np
import scanpy as _sc
import decoupler as _dc
import pandas as _pd

_adata_enr = adata.copy()
_adata_enr.X = _adata_enr.layers['counts'].copy() if 'counts' in _adata_enr.layers else _adata_enr.X.copy()
_sc.pp.normalize_total(_adata_enr, target_sum=1e4)
_sc.pp.log1p(_adata_enr)
_X = _adata_enr.X.toarray() if hasattr(_adata_enr.X, 'toarray') else _np.asarray(_adata_enr.X)
_adata_enr.X = _np.nan_to_num(_X, nan=0.0, posinf=0.0, neginf=0.0)
print('Enrichment adata:', _adata_enr.shape)

# Run decoupler ULM directly.
_dc.mt.ulm(data=_adata_enr, net=progeny_net, tmin=5, verbose=False)
_score_adata = _dc.pp.get_obsm(_adata_enr, key='score_ulm')
_vals = _score_adata.X
if hasattr(_vals, 'toarray'):
    _vals = _vals.toarray()
_vals = _np.asarray(_vals)
_program_cols = [f'ulm__{c}' for c in _score_adata.var_names]

# Reorder rows from adata-order to adata_spv-order using per-group integer positions.
_perm = _np.concatenate([
    _np.where(adata.obs['antigen_specific'].values == _g)[0]
    for _g in sorted(adata.obs['antigen_specific'].unique())
])
assert _perm.shape[0] == adata_spv.n_obs, (_perm.shape, adata_spv.n_obs)
scores_df = _pd.DataFrame(_vals[_perm], index=adata_spv.obs_names, columns=_program_cols)
adata_spv.obsm['X_spvm_enrichment'] = scores_df.values
adata_spv.uns['spvm_enrichment'] = {'columns': _program_cols, 'method': 'ulm'}
print('Enrichment scores:', scores_df.shape)
print('Programs:', _program_cols[:6], '...')


In [ ]:
summary = model_spv.summarize_enrichment(scores_df, groupby="cluster_label", agg="mean")
summary.iloc[:, :6]

In [ ]:
fig = spVIPESmulti.pl.enrichment_heatmap(summary, top_n=10)

In [ ]:
report = model_spv.interpretation_report(
    scores_df,
    groupby="cluster_label",
    z_shared_key="X_spvm_shared",
    label_key="cluster_label",
    top_n=5,
)
print("Top programs per cluster:")
for grp, progs in report["top_programs"].items():
    print(f"  {grp}: {progs}")
if report.get("integration_metrics") is not None:
    print("\nIntegration metrics from report:")
    print(report["integration_metrics"])

In [ ]:
fig = spVIPESmulti.pl.interpretation_dashboard(
    adata_spv,
    scores_df,
    groupby="cluster_label",
    shared_basis="X_umap_spvm_shared",
    top_n=10,
)

## 17. Reproducibility footer

In [ ]:
import datetime
print(f"Run date      : {datetime.date.today()}")
print(f"spVIPESmulti  : {spVIPESmulti.__version__}")
print(f"scvi-tools    : {scvi.__version__}")
print(f"scanpy        : {sc.__version__}")
print(f"torch         : {torch.__version__}")